# 01 — Cleaning and Anonymization

**Inputs**:
- `data/raw/survey_responses.xlsx` — Brazilian PT-BR form (32 responses, 63 columns)
- `data/raw/survey_responses_2.xlsx` — international EN form (9 responses, 62 columns)

**Outputs**:
- `data/processed/anonymized.csv` — unified dataset (n=41), no PII, stable schema
- `data/processed/likert_importance.csv` — 13 characteristics × 41 respondents (long)
- `data/processed/likert_priority.csv` — same for priority
- `data/processed/skills.csv` — 10 activities × 41 respondents
- `data/processed/open_responses.csv` — all open responses in long form
- `data/codebook/codebook.md` — documented schema (English)

**Quality criteria**:
- Confirm 41 rows (32 PT + 9 EN) × 63 columns (incl. `language`)
- Validate informed-consent in every response (PT/EN)
- Drop emails + `@dropdown` Google Forms artefact
- Free text kept in original language; ordinal Likert/skill/freq mapped via `utils.py` (bilingual keys)
- Split country×UF; non-BR respondents get `region='International'`

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import utils as U

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## 1. Loading + shape validation

In [2]:
df = U.load_raw()
assert df.shape == (41, 63), df.shape
print(f"Shape: {df.shape}")
print(f"Subset by language: {df['language'].value_counts().to_dict()}")
df.head(3)

Shape: (41, 63)
Subset by language: {'pt': 32, 'en': 9}


,language,timestamp,consent,age,state,gender,education,role,seniority,n_projects,skill_cleaning,skill_normalization,skill_outliers,skill_integration,skill_transformation,skill_validation,skill_pipelines,skill_monitoring,skill_libs,skill_split,word_1,word_2,word_3,word_4,word_5,re_experience,imp_precision,imp_completeness,imp_consistency,imp_credibility,imp_currentness,imp_accessibility,imp_compliance,imp_reliability,imp_efficiency,imp_traceability,imp_understandability,imp_availability,imp_recoverability,imp_justification,pri_precision,pri_completeness,pri_consistency,pri_credibility,pri_currentness,pri_accessibility,pri_compliance,pri_reliability,pri_efficiency,pri_traceability,pri_understandability,pri_availability,pri_recoverability,pri_justification,balance_open,versioning_open,incorporation_open,measurement_open,discussion_freq,documentation_open,challenges_open,support_freq,_email_drop
0,pt,2025-02-14 15:15:25.144,Aceito participar,18-24 anos,Ceará,Homem,Ensino superior,Cientista de dados,Júnior (até 5 anos),2,Média,Média,Média,Média,Média,Média,Média,Média,Média,Média,Precisão,Consistência,Completude,Relevância,Viés,"Sim, foi uma ótima experiência, embora complic...",Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,NaN,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,NaN,"Equilibrar precisão, compreensibilidade e efic...",Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Menos de uma vez por semana, mas pelo menos um...","Linguagem estruturada (texto), Ferramentas de ...",Inconsistência entre diferentes fontes de dado...,Ocasionalmente,NaN
1,pt,2025-02-14 15:46:31.820,Aceito participar,25-34 anos,PE,Mulher,Estudante de Doutorado,Cientista de dados,Júnior (até 5 anos),2,Muito alto,Muito alto,Muito alto,Média,Acima da média,Acima da média,Média,Média,Acima da média,Muito alto,Fonte,Estabilidade,Necessário,Essencial,Determinante,Não,Muito importante,Importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Importante,Muito importante,Muito importante,Muito importante,NaN,Essencial,Alta prioridade,Essencial,Essencial,Essencial,Essencial,Essencial,Essencial,Essencial,Alta prioridade,Essencial,Essencial,Alta prioridade,NaN,"A prioridade é na precisão dos dados, seguido ...",Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Pelo menos uma vez por semana, mas não todos o...","Linguagem estruturada (texto), Reuniões de Ali...","Dados incompletos ou ausentes, Dados desatuali...",Ocasionalmente,NaN
2,pt,2025-02-14 15:53:04.364,Aceito participar,25-34 anos,Ceará,Homem,Mestrado,Cientista de dados,Pleno (6 a 9 anos),13,Muito alto,Muito alto,Muito alto,Acima da média,Muito alto,Muito alto,Média,Média,Acima da média,Muito alto,Confiabilidade,Disponibilidade,Atualização,Integridade,Processamento,"Geralmente, são os primeiros passos quando co...",Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,NaN,Alta prioridade,Alta prioridade,Alta prioridade,Essencial,Essencial,Essencial,Essencial,Essencial,Neutro,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,NaN,Tudo vai depender de como nossa aplicação prec...,Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Menos de uma vez por semana, mas pelo menos um...","Linguagem estruturada (texto), Ferra

## 2. Informed-consent validation

All 41 respondents must have agreed to the informed-consent form (PT: "Aceito participar"; EN: "I agree to participate"). Drop anyone who did not.

In [3]:
consent_counts = df["consent"].value_counts(dropna=False)
print(consent_counts)
CONSENT_OK = {"Aceito participar", "I agree to participate"}
assert df["consent"].isin(CONSENT_OK).all(), "Some respondent did not agree to informed consent!"
print(f"\n[OK] All {len(df)} respondents agreed to informed consent.")

consent
Aceito participar         32
I agree to participate     9
Name: count, dtype: int64

[OK] All 41 respondents agreed to informed consent.


## 3. Duplicate detection (timestamp + profile)

In [4]:
n_dup_timestamp = df["timestamp"].duplicated().sum()
n_dup_profile = df.duplicated(subset=["age", "state", "gender", "role", "seniority", "n_projects"]).sum()
print(f"Duplicate timestamps: {n_dup_timestamp}")
print(f"Duplicate demographic profiles: {n_dup_profile}")
print(f"Collection window: {df['timestamp'].min()} → {df['timestamp'].max()}")

Duplicate timestamps: 0
Duplicate demographic profiles: 1
Collection window: 2025-02-14 15:15:25.144000 → 2025-03-15 16:34:41.757000


## 4. Drop PII / empty columns

- `_email_drop` — email column (Q23) — anonymization (PT + EN)
- The `@dropdown` column (Forms artefact in PT) was already removed in `load_raw`

In [5]:
n_emails = df["_email_drop"].notna().sum()
print(f"Emails provided (will be removed): {n_emails}/{len(df)}")
df = df.drop(columns=["_email_drop", "consent"])
print(f"Shape after drops: {df.shape}")

Emails provided (will be removed): 20/41
Shape after drops: (41, 61)


## 5. Country and UF normalization

PT respondents only filled in their state; EN respondents wrote "Country" or "State, Country" freely. Here we derive:

- `country` — canonical country (`Brazil`, `Germany`, `France`, etc.)
- `state` — 2-letter UF code when country == 'Brazil', else NaN
- `region` — Brazilian macro-region or `International`

In [6]:
df["state_raw"] = df["state"]
parsed = df["state_raw"].apply(U.parse_country_state)
df["country"] = parsed.map(lambda t: t[0])
df["state"] = parsed.map(lambda t: t[1])
df["region"] = df.apply(lambda r: U.country_to_region(r["country"], r["state"]), axis=1)

print("country × UF × region mapping:")
print(df[["country", "state", "region"]].value_counts(dropna=False).to_string())

unmapped = df[df["country"].isna()]
if len(unmapped):
    print("\n[WARNING] Unidentified countries:", unmapped["state_raw"].tolist())
else:
    print("\n[OK] All respondents mapped to a country.")

country × UF × region mapping:
country   state  region       
Brazil    CE     Northeast        17
          RJ     Southeast         5
          SP     Southeast         3
Germany   NaN    International     3
Brazil    AL     Northeast         2
          BA     Northeast         2
France    NaN    International     2
Brazil    PE     Northeast         1
          PR     South             1
          RS     South             1
          PB     Northeast         1
          AM     North             1
Colombia  NaN    International     1
Brazil    NaN    NaN               1

[OK] All respondents mapped to a country.


## 6. Numeric coercions and Likert mapping

- `n_projects`: PT comes as `int`; EN sometimes brings `string` (e.g. `"7-10, mostly university projects"`). Coerce to range midpoint, fallback to first number extracted.
- Skill Q8: 1 (Very low) → 5 (Very high), "Not applicable" → NaN
- Importance Q11: 1 (Not important) → 5 (Very important)
- Priority Q13: 1 (No priority) → 5 (Essential)
- Discussion Q19: 1 (Never) → 5 (Every day)
- Support Q22: 1 (Rarely) → 4 (Always)
- Normalized demographics: `gender_norm`, `age_band`, `education_norm` for cross-language analyses

Original columns kept as `*_raw` for auditability.

In [7]:
def apply_map(df: pd.DataFrame, cols: list[str], mapping: dict, suffix: str = "_raw") -> pd.DataFrame:
    """Rename col -> col_raw and create new col with mapping applied. Fails if any value is missing from mapping."""
    for c in cols:
        df[c + suffix] = df[c]
        unknown = set(df[c].dropna().unique()) - set(mapping.keys())
        if unknown:
            raise ValueError(f"Unexpected values in {c}: {unknown}")
        df[c] = df[c].map(mapping).astype("Int64")
    return df


def coerce_n_projects(value):
    """PT: int directly. EN: some come as strings (e.g. '7-10, mostly university projects')."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float)):
        return int(value)
    s = str(value)
    nums = re.findall(r"\d+", s)
    if not nums:
        return pd.NA
    nums = [int(x) for x in nums]
    if len(nums) >= 2 and "-" in s:  # range "7-10"
        return int(round((nums[0] + nums[1]) / 2))
    return nums[0]


df["n_projects_raw"] = df["n_projects"]
df["n_projects"] = df["n_projects_raw"].apply(coerce_n_projects).astype("Int64")

df = apply_map(df, U.SKILL_COLS, U.SKILL_MAP)
df = apply_map(df, U.IMP_COLS, U.IMPORTANCE_MAP)
df = apply_map(df, U.PRI_COLS, U.PRIORITY_MAP)
df = apply_map(df, ["discussion_freq"], U.DISCUSSION_FREQ_MAP)
df = apply_map(df, ["support_freq"], U.SUPPORT_FREQ_MAP)

df["seniority_ordinal"] = df["seniority"].map(U.SENIORITY_ORDINAL).astype("Int64")
df["seniority_group"] = df["seniority"].map(U.SENIORITY_GROUP)
df["role_group"] = df["role"].map(U.ROLE_GROUP)

# Language-agnostic demographics
df["gender_norm"] = df["gender"].map(U.GENDER_NORM)
df["age_band"] = df["age"].map(U.AGE_BAND)
df["education_norm"] = df["education"].map(U.EDUCATION_NORM)

assert df["seniority_ordinal"].notna().all(), "Seniority not mapped"
assert df["role_group"].notna().all(), f"Role not mapped: {df[df['role_group'].isna()]['role'].unique()}"
assert df["gender_norm"].notna().all(), f"Gender not mapped: {df[df['gender_norm'].isna()]['gender'].unique()}"
assert df["age_band"].notna().all(), f"Age band not mapped: {df[df['age_band'].isna()]['age'].unique()}"
assert df["education_norm"].notna().all(), f"Education not mapped: {df[df['education_norm'].isna()]['education'].unique()}"

print("Likerts mapped. Sanity check (importance of precision):")
print(df["imp_precision"].value_counts(dropna=False).sort_index())
print("\nSeniority group:")
print(df["seniority_group"].value_counts(dropna=False))
print("\nRole group:")
print(df["role_group"].value_counts(dropna=False))

Likerts mapped. Sanity check (importance of precision):
imp_precision
1     1
3     1
4     8
5    31
Name: count, dtype: Int64

Seniority group:
seniority_group
senior    22
junior    19
Name: count, dtype: int64

Role group:
role_group
data_scientist    22
developer         11
ml_engineer        2
data_engineer      2
other              2
manager            1
researcher         1
Name: count, dtype: int64


## 7. Open-response inspection

How many non-empty responses per question, plus a sweep for proper-name mentions (potential identification risk).

In [8]:
OPEN_COLS = [
    "re_experience", "imp_justification", "pri_justification",
    "balance_open", "versioning_open", "incorporation_open",
    "measurement_open", "documentation_open", "challenges_open",
]
summary = pd.DataFrame({
    "question": OPEN_COLS,
    "n_responses": [df[c].notna().sum() for c in OPEN_COLS],
    "avg_chars": [int(df[c].dropna().str.len().mean()) if df[c].notna().any() else 0 for c in OPEN_COLS],
    "max_chars": [int(df[c].dropna().str.len().max()) if df[c].notna().any() else 0 for c in OPEN_COLS],
})
print(summary.to_string(index=False))

          question  n_responses  avg_chars  max_chars
     re_experience           41        167       1009
 imp_justification           15        237        642
 pri_justification            9        228        623
      balance_open           40        285       1733
   versioning_open           41        129        138
incorporation_open           41        156        253
  measurement_open           41         83        543
documentation_open           41         94        220
   challenges_open           41        156        344


In [9]:
# Proper-name mention detection. Simple heuristic: emails and tokens with an
# initial capital that are not at the start of a sentence. False positives are
# tolerated — output is for manual review only.
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PROPER_NAME_RE = re.compile(r"(?<![\.!?]\s)(?<!^)\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]{2,})\b")

STOP_PROPER = {
    # Items that appear capitalized but are not proper names
    "Brasil", "Brazil", "Python", "Java", "JavaScript", "SQL", "Pandas", "PySpark", "Dask",
    "PCA", "Excel", "Power", "Bi", "Microsoft", "Google", "AWS", "GCP", "Azure",
    "Linux", "Mac", "Windows", "Github", "Git", "Gitlab", "Bitbucket", "Jira",
    "Confluence", "Slack", "Teams", "Zoom", "Notion", "Tableau",
    "Apache", "Spark", "Hadoop", "Kafka", "Airflow", "Mlflow", "TensorFlow",
    "PyTorch", "Sklearn", "Scikit", "Numpy", "Scipy", "Matplotlib",
    "Tcle", "Ml", "Ia", "Br", "Ex", "Etl", "Api", "Crud", "Mvp", "Poc",
    "Engenharia", "Requisitos", "Projeto", "Equipe", "Cliente", "Aprendizado",
    "Maquina", "Máquina", "Dados", "Qualidade", "Modelo", "Modelos",
}

alerts = []
for col in OPEN_COLS:
    for idx, txt in df[col].dropna().items():
        emails = EMAIL_RE.findall(txt)
        proper = [m for m in PROPER_NAME_RE.findall(txt) if m not in STOP_PROPER]
        if emails or proper:
            alerts.append({
                "row": idx,
                "col": col,
                "emails": emails,
                "proper_name_candidates": proper[:5],
                "snippet": txt[:200],
            })

alerts_df = pd.DataFrame(alerts)
print(f"Alerts for manual review: {len(alerts_df)}")
if len(alerts_df):
    print(alerts_df.to_string())

Alerts for manual review: 138
     row                 col emails                                               proper_name_candidates                                                                                                                                                                                                    snippet
0      4       re_experience     []                                                            [Ciência]                                                                                                                                Já participei de todas as fases do ciclo de um projeto em Ciência de Dados.
1     29       re_experience     []                   [Histórias, Cenários, Funcionais, Não, Funcionais]   Participei na melhoria de requisitos do meu projeto atual. Nesse processo sempre é feita uma reunião com o cliente onde ele faz o levantamento dos cenários e casos que ele deseja obter. Ao final dessa
2     32       re_experience     []           

## 8. 5-words inspection (Q9)

Very short text — normalize lowercase + strip. Report raw counts (lemmatization happens in notebook 04).

In [10]:
for c in U.WORD_COLS:
    df[c + "_raw"] = df[c]
    df[c] = df[c].astype("string").str.strip().str.lower().replace({"": pd.NA})

word_long = df[U.WORD_COLS].melt(value_name="word", var_name="position").dropna()
word_long["position"] = word_long["position"].str.replace("word_", "").astype(int)
print(f"Total tokens: {len(word_long)} (max possible 41×5=205)")
print("Top-15 raw:")
print(word_long["word"].value_counts().head(15))

Total tokens: 204 (max possible 41×5=205)
Top-15 raw:
word
consistência       10
precisão            7
relevância          7
completude          7
confiabilidade      6
atualização         6
qualidade           4
quantidade          4
volume              4
outliers            3
acurácia            3
balanceamento       2
corretude           2
limpeza             2
disponibilidade     2
Name: count, dtype: Int64


## 9. Quality report (missing per column, effective n per question)

In [11]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing.to_string())

Columns with missing values:
pri_justification       32
imp_justification       26
state                    7
skill_monitoring         3
skill_pipelines          2
skill_outliers           2
skill_normalization      2
skill_transformation     2
skill_cleaning           2
skill_integration        2
skill_split              1
word_2                   1
skill_libs               1
support_freq_raw         1
support_freq             1
region                   1
balance_open             1
skill_validation         1
word_2_raw               1


In [12]:
# Cross-tab sanity check: age × seniority × role
print("Age × Seniority:")
print(pd.crosstab(df["age_band"], df["seniority_group"]))
print("\nRole × number of projects (mean):")
print(df.groupby("role_group")["n_projects"].agg(["count", "mean", "median"]).round(1))

Age × Seniority:
seniority_group  junior  senior
age_band                       
18-24                 6       0
25-34                11      12
35-44                 2       6
45-54                 0       4

Role × number of projects (mean):
                count  mean  median
role_group                         
data_engineer       2   4.0     4.0
data_scientist     22   8.1     4.5
developer          11   4.5     3.0
manager             1  10.0    10.0
ml_engineer         2   4.5     4.5
other               2   2.5     2.5
researcher          1   4.0     4.0


## 10. Persistence

Save 5 separate CSVs so downstream notebooks can consume them without re-processing.

In [13]:
out = U.DATA_PROC / "anonymized.csv"
df.to_csv(out, index=False)
# print(f"[saved] {out} — {df.shape}")

In [14]:
# Long forms for aggregated analyses
imp_long = df[U.IMP_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="characteristic", value_name="importance"
).dropna()
imp_long["characteristic"] = imp_long["characteristic"].str.replace("imp_", "")
imp_long.to_csv(U.DATA_PROC / "likert_importance.csv", index=False)
print(f"[saved] likert_importance.csv — {imp_long.shape}")

pri_long = df[U.PRI_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="characteristic", value_name="priority"
).dropna()
pri_long["characteristic"] = pri_long["characteristic"].str.replace("pri_", "")
pri_long.to_csv(U.DATA_PROC / "likert_priority.csv", index=False)
print(f"[saved] likert_priority.csv — {pri_long.shape}")

skills_long = df[U.SKILL_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="activity", value_name="skill"
)
skills_long.to_csv(U.DATA_PROC / "skills.csv", index=False)
print(f"[saved] skills.csv — {skills_long.shape}")

open_long = df[OPEN_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="question", value_name="response"
).dropna()
open_long.to_csv(U.DATA_PROC / "open_responses.csv", index=False)
print(f"[saved] open_responses.csv — {open_long.shape}")

word_long.to_csv(U.DATA_PROC / "words.csv", index=False)
print(f"[saved] words.csv — {word_long.shape}")

[saved] likert_importance.csv — (533, 3)
[saved] likert_priority.csv — (533, 3)


[saved] skills.csv — (410, 3)


[saved] open_responses.csv — (310, 3)
[saved] words.csv — (204, 2)


## 11. Codebook — documented schema

Generated automatically from the lookup tables in `utils.py`.

In [15]:
lines = ["# Codebook — Data Quality Requirements in ML Survey", ""]
n_pt = int((df["language"] == "pt").sum())
n_en = int((df["language"] == "en").sum())
lines.append(f"**N**: {len(df)} respondents ({n_pt} from PT-BR form + {n_en} from EN international form)")
lines.append(f"**Collection window**: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
lines.append("")
lines.append("> Free-text responses and demographic categoricals are kept in their original "
             "language (PT or EN). For cross-language analyses use the derived columns: "
             "`gender_norm`, `age_band`, `education_norm`, `seniority_group`, `role_group`. "
             "Likert items (Q8/Q11/Q13/Q19/Q22) are mapped to identical ordinal scales across "
             "both subsets via bilingual lookup tables in `notebooks/utils.py`.")
lines.append("")
lines.append("## Schema")
lines.append("")
lines.append("| Column | Type | Domain | Source (Q) |")
lines.append("|---|---|---|---|")
DOMAIN = {
    "language": ("categorical", "pt, en", "source subset"),
    "timestamp": ("datetime", "Forms timestamp", "-"),
    "age": ("categorical", "raw label (PT \"anos\" / EN \"years old\")", "Q1"),
    "age_band": ("categorical", "18-24 / 25-34 / 35-44 / 45-54 / ...", "derived"),
    "country": ("categorical", "Brazil, Germany, France, Colombia, ...", "derived (Q2)"),
    "state": ("UF (2 letters)", "AC..TO (only when country=Brazil)", "Q2"),
    "region": ("categorical", "North / Northeast / Central-West / Southeast / South / International", "derived"),
    "gender": ("categorical", "raw label (PT or EN)", "Q3"),
    "gender_norm": ("categorical", "male, female, other, undisclosed", "derived"),
    "education": ("categorical", "raw label (PT or EN)", "Q4"),
    "education_norm": ("categorical",
                        "high_school / undergraduate / ms_student / master / phd_student / doctorate / specialization",
                        "derived"),
    "role": ("categorical", "raw label (PT or EN)", "Q5"),
    "role_group": ("categorical",
                    "data_scientist / developer / ml_engineer / data_engineer / researcher / manager / other",
                    "derived"),
    "seniority": ("ordinal", "Intern / Junior / Mid / Senior (PT or EN label)", "Q6"),
    "seniority_ordinal": ("int 1-4", "1=Intern .. 4=Senior", "derived"),
    "seniority_group": ("categorical", "junior, senior", "derived"),
    "n_projects": ("int", "0..40 (EN ranges parsed to midpoint)", "Q7"),
}
for col, (t, dom, q) in DOMAIN.items():
    lines.append(f"| `{col}` | {t} | {dom} | {q} |")
for c in U.SKILL_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Very low .. 5=Very high | Q8 |")
for c in U.WORD_COLS:
    lines.append(f"| `{c}` | string | free token, lowercased | Q9 |")
for c in U.IMP_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Not important .. 5=Very important | Q11 |")
for c in U.PRI_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=No priority .. 5=Essential | Q13 |")
for c in ("re_experience", "imp_justification", "pri_justification",
          "balance_open", "versioning_open", "incorporation_open",
          "measurement_open", "documentation_open", "challenges_open"):
    lines.append(f"| `{c}` | string (open) | free-text response in PT or EN | varies |")
lines.append("| `discussion_freq` | Likert 1-5 | 1=Never .. 5=Every day | Q19 |")
lines.append("| `support_freq` | Likert 1-4 | 1=Rarely .. 4=Always | Q22 |")

lines.append("")
lines.append("## Anonymization")
lines.append("")
lines.append(f"- Email column (Q23): dropped from both forms. {n_emails} respondents provided an email overall. Not redistributed.")
lines.append("- `@dropdown` column: empty Google Forms artefact (PT only). Removed in `load_raw`.")
lines.append("- `country` × `state`: Brazilian respondents normalized to 2-letter UF codes; non-BR respondents (Germany, France, Colombia) keep `country` only with `region='International'` — no identification beyond country level.")
lines.append("- Open responses: regex sweep for emails and proper-name candidates. No personal identifier was found. See cell 7 of notebook 01 for the procedure.")

(U.DATA_CODEBOOK / "codebook.md").write_text("\n".join(lines))
print(f"[saved] codebook.md — {len(lines)} lines")

[saved] codebook.md — 87 lines


## 12. Final summary

In [16]:
print(f"Final N: {len(df)}")
print(f"Final columns: {df.shape[1]}")
print(f"Columns with any missing: {(df.isna().sum() > 0).sum()}")
print(f"Non-empty Q9 tokens: {len(word_long)}")
print(f"Non-empty open responses: {len(open_long)}")

Final N: 41
Final columns: 114
Columns with any missing: 19
Non-empty Q9 tokens: 204
Non-empty open responses: 310
